# Домашнее задание: Сбор данных и разметка: от формулировки задачи до крауда

В этом кейсе вы пройдёте путь **от постановки бизнеса** до **пайплайна гибридной разметки**:  
Постановка задачи → разметка zero shot промптом с помощью LLM → оценка качества → Улучшение качества промпта: few-shot, cot и другие способы → Оценка уверенности ответа




### Установка зависимостей

In [ ]:
%pip install "torch>=2.6,<3" "transformers==4.57.1" "datasets>=4,<5" accelerate scikit-learn pandas tqdm matplotlib


In [ ]:
import torch, json, random, re, pandas as pd, numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from sklearn.metrics import precision_recall_fscore_support
from tqdm.auto import tqdm
torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu'
print('Device:', device)


## 1. Постановка задачи


**Контекст (от лица бизнеса):**
Наша компания разрабатывает финтех-приложение с поддержкой пользователей через чат-бот.
Мы хотим автоматически определять тему запроса клиента (например: "блокировка карты", "потеря ПИН-кода", "перевыпуск карты", и т.д.), чтобы быстро направлять клиента к нужному решению. Требуется получить данные для задачи




**Описание задачи:**
> "Для каждого входящего текстового сообщения пользователя автоматически определить одну из тематик (например, balance, card_not_working, transfer, etc.)"

## 2. Требования и бизнес-метрики – 1 балл

Предложите не менее 2 бизнес-метрик, которые может хотеть оптимизировать бизнес относительно процесса разметки данных для данной задачи.


In [ ]:
# ваш ответ тут

# ---- Ваш код здесь ----
print("""
Хорошими метриками будет X, Y и Z, потому что так и есть
""")
# ---- Конец кода ----

## 3. Сведение к ML-задаче – 2 балла



Сведите бизнес-задачу к задаче машинного обучения, опишите входные данные и метки:

- **Тип задачи**:
- **Объект**:
- **Метки**:

In [ ]:
# ваш ответ тут

# ---- Ваш код здесь ----
print("""
- **Тип задачи**:
- **Объект**:
- **Метки**:
""")
# ---- Конец кода ----

## 4. ML-метрики – 2 балла


Сформулируйте, какие метрики вы будете отслеживать в процессе сбора данных и получения разметки: как при помощи LLM, так и при помощи разметчиков в крауде

In [ ]:
# ---- Ваш код здесь ----
print("""
- LLM разметчик: (метрики)
- Разметчики в крауде: (метрики)

""")
# ---- Конец кода ----



## 5. Данные и бейзлайн разметка

### 5.1 Загрузка и первичный анализ датасета

Посмотрим данные: примеры из датасета (попробуем разметить сами хотя бы 10 примеров), все типы меток, размеры выборок, распределение

In [ ]:
# CSV из репозитория авторов: загрузчик не требует устаревшего dataset script.
# Источник и лицензия CC BY 4.0: https://github.com/PolyAI-LDN/task-specific-datasets/tree/master/banking_data
from datasets import load_dataset, ClassLabel
import pandas as pd

DATA_REVISION = "57ec275d8078af65b7731c2a98be812d844a6d6b"
DATA_URL = f"https://raw.githubusercontent.com/PolyAI-LDN/task-specific-datasets/{DATA_REVISION}/banking_data"
ds = load_dataset("csv", data_files={split: f"{DATA_URL}/{split}.csv" for split in ("train", "test")})
ds = ds.rename_column("category", "label")
label_names = sorted(set(ds["train"]["label"]))
assert len(label_names) == 77
assert set(ds["test"]["label"]) == set(label_names)
ds = ds.cast_column("label", ClassLabel(names=label_names))
# Сохраняем ds["train"], ds["test"], text и числовой label, как у исходного загрузчика.
# Соответствие ID -> метка задаёт ds["train"].features["label"].names; не берите ID из других версий.


# ---- Ваш код здесь ----
print("""
Считываем данные
""")
# ---- Конец кода ----


### 5.2 Бейзлайн LLM разметка (7 баллов)

В этом пункте нужно получить бейзлайн разметку с помощью open source LLM и простого короткого промпта.

Для упрощения тут у нас уже есть golden set разметка (в случае если не было бы, то действовали как указано в лекции, или бы размечали для начала сами хотя бы 50-100 примеров), на которой мы можем проверять качество

Для старта используйте небольшую модель `Qwen/Qwen3-1.7B` с отключённым thinking (`enable_thinking=False`). Она запускается через Transformers на Apple Silicon (MPS, macOS 14+), NVIDIA GPU (CUDA) или CPU; на CPU работа может быть медленнее. Квантизация и bitsandbytes для этого варианта не нужны. Для быстрой отладки можно взять `Qwen/Qwen3-0.6B`; для итоговой оценки используйте модель, которая достигает заданного качества. Далее в следующих ячейках для улучшения можете использовать модели размера больше


In [ ]:
# Для базовой модели достаточно зависимостей из начала ноутбука.


In [ ]:
import time

MODEL_NAME = "Qwen/Qwen3-1.7B"
MODEL_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
device = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
# На GPU уменьшаем память весов; на CPU используем float32.
model_dtype = torch.bfloat16 if device == "mps" else torch.float16 if device == "cuda" else torch.float32
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, revision=MODEL_REVISION)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, revision=MODEL_REVISION, dtype=model_dtype,
).to(device).eval()
print(f"Модель: {MODEL_NAME}; устройство: {device}; dtype: {model_dtype}")
print("Для замены модели измените MODEL_NAME и задайте её revision либо None.")

# Модель и tokenizer готовы. Ниже реализуйте промпт, генерацию и парсинг.
# Для сообщений используйте tokenizer.apply_chat_template(..., enable_thinking=False).


Формируем простой короткий промпт, в котором укажем  все категории меток для разметки, и зададим нужный формат ответа: например, json {"label": str} или только название метки. Для выбранного формата напишем функцию парсинга


В примере решения разрешён также один Markdown-блок `json` вокруг JSON. Произвольный текст вокруг ответа не принимается; допустимый формат должен быть указан в промпте и одинаково проверяться во всех экспериментах.


In [ ]:
# ---- Ваш код здесь ----
prompt_template = (
    "Текст промпта для разметки"
    )
# ---- Конец кода ----


Делаем разметку 10-20 примеров, пишем функцию парсинга ответа (считаем метрику в скольких ответах нарушения следования формату), смотрим ответы
На выходе покажите таблицу из 10–20 строк: текст, истинная метка, предсказанная метка, исходный ответ и признак ошибки формата.


In [ ]:
# Функция разметки вместе с промптом,

# ---- Ваш код здесь ----
print("""
    прокачиваем в цикле выбранную LLM для разметки данных через функцию annotate, добавляем разметку в исходный датасет и сохраняем в файл
""")
# ---- Конец кода ----


### 5.3 Оценка качества (2 балла)

Оцениваем качество разметки на тестовом датасет (либо на семпле из тестового датасета)


На выходе: accuracy, доля ошибок формата и число оценённых примеров. Непарсящиеся ответы считаем ошибками, а не исключаем из выборки.

Для сравнения вариантов используйте одни и те же 100 примеров из test с random_state=2024. 10–20 примеров оставьте для отладки формата. На такой небольшой выборке малые различия метрик не стоит переоценивать.


In [ ]:
# ---- Ваш код здесь ----
print("""
    Инферим LLM на тесте, замеряем метрики
""")
# ---- Конец кода ----



### 6. Улучшение качества промпта

### 6.1 few shot prompt (4 баллов)

Добавим в промпт few-shot примеры (важно, чтобы не было data leak  c тестом): помогает исправлять поведение модели, когда описание в инструкции не справляется + модель лучше следует форматам
Вместо few-shot можно проверить другой способ улучшения инструкции: объясните выбор и сравните качество на тех же примерах.


In [ ]:
# ---- Ваш код здесь ----
print("""
    Пишем промпт с few shot, замеряем качество
""")
# ---- Конец кода ----



### 6.2 chain-of-thoughts (3 балла)

Пробуем добавить сhain-of-thought в промпт: просим короткий reasoning и проверяем, помогает ли он на более сложных задачах. Объяснение модели не гарантирует правильность ответа и не обязательно отражает реальные причины решения.

Если используете JSON с reasoning, ответ может быть такого формата
{"reasoning": "why_this_class", "label": "one_of_the_categories"}
Вместо reasoning можно проверить другую гипотезу по ошибкам модели, отличающуюся от предыдущего пункта. Покажите изменение промпта, качество до/после и короткий вывод.


In [ ]:
# ---- Ваш код здесь ----
print("""
    Пишем промпт с reasoning или другим выбранным улучшением, замеряем качество
""")
# ---- Конец кода ----



### 6.3 Дальнейшие улучшения (6 баллов)

Далее улучшаем итеративно

Основные улучшения в общем случае происходит за счет:
- Аналитика ошибок,  в первую очередь анализируем ошибки разметки (только на трейне, чтобы не подогнаться под тест!), в том числе используя reasoning модели, чтобы понять причины. Также помогает спрашивать у самой модели и просить ее поправить начальный промпт/инструкцию

- Понимание бизнеса и домена, четкое описание в инструкции/промпте

Дополнительно, что тут может еще помочь:
- Упрощение задачи: размечать не одну, а несколько наиболее релеватных меток для каждого текста (=> растим recall)

- Использовать более "умные" LLM
- Размечаем с перекрытием: запускаем промпт n раз (например, 3) и агрегируем ответ. Улучшение: агрегурем результат ансамбля разных LLM (среди тех же размеров например: Qwen-3 8b) , можно с тем же промптом, либо промпты могут отлчичаться между собой few shot примерами

Тут нужно реализовать одно из улучшений (из лекции: слайды про улучшение 39-41, 46, либо списка выше, например, с перекрытием). Цель — получить итоговую accuracy ≥ 0.6 на тестовой выборке и сравнить её с исходным baseline.
Если текущая модель не достигает 0.6, проверьте более сильную модель — это допустимый способ улучшения. Если baseline уже выше 0.6, всё равно покажите осмысленный эксперимент и анализ: прирост от каждой отдельной техники не гарантирован. При смене модели укажите это в сравнении.


In [ ]:
# ---- Ваш код здесь ----
print("""
    Промпты с улучшением качества (baseline: accuracy = 0.6), сравнение метрик, как каждое улучшение повляило
""")
# ---- Конец кода ----

## 7.1. Уверенность ответа. (11 баллов)

Посчитаем уверенность модели ответов по модели. Полезно для гибкой схемы разметки, когда более сложные примеры отправляются на разметку асессорам, либо на модель побольше, или на доп разметку с доп перекрытием.

Рабочий бейзлайн — exp(mean(logprob)) только по токенам итоговой метки. Не включаем reasoning, JSON-ключи и скобки, промпт и EOS. Это score для ранжирования ответов, а не вероятность правильности класса.

Здесь нужно написать функцию для взятия уверенности модели, как указано ниже, и далее показать, что числовая «уверенность» (confidence), посчитанная по лог-вероятностям токенов, действительно коррелирует с тем, ошиблась модель или нет. Ожидается получение AUC>=0.6


https://cookbook.openai.com/examples/using_logprobs

Для reasoning-модели проще начать с отключённого thinking; у Qwen3 это enable_thinking=False в tokenizer.apply_chat_template. Если получаете метку отдельным вызовом, оценивайте именно метку и confidence этого вызова. Покажите 3 примера: ответ → метка → выбранные токены → confidence. Если все ответы правильные или все неправильные, ROC-AUC не определён — укажите это.

Для базового решения достаточно отдельного вызова с ответом только в виде метки — извлекать токены метки из JSON или reasoning не обязательно. Покажите ROC-AUC для всех ответов и отдельно для валидных меток: нулевой confidence у ошибок формата сам по себе может улучшать общий AUC. Цель 0.6 — ориентир; корректный отрицательный результат с объяснением тоже принимается.


In [ ]:

# ---- Ваш код здесь ----
def annotate_conf(text: str,
                  max_new_tokens: int = 32
                 ) -> tuple[str | None, float, int, str]:
    """
    Размечает один запрос при помощи LLM и сразу возвращает числовую
    «уверенность» предсказания на основе лог-вероятностей сгенерированных
    токенов.

    ▸ Логика шага
      1. Формируем prompt (например с few-shot).
      2. Вызываем `model.generate(..., output_scores=True, return_dict_in_generate=True)` — получаем
         логиты (score-векторы) для каждого сгенерированного токена.
      3. Извлекаем label из выбранного формата ответа; если он не распарсился
         или не входит в `label_names`, возвращаем label=None, confidence=0, corrupted=1.
      4. Преобразуем logits в logprobs и сопоставляем с токенами генерации.
         **Confidence** = `exp(mean(log p))` только по токенам итоговой метки,
         без reasoning, JSON-обвязки, промпта и EOS. Это score, не вероятность
         правильности класса; вероятность после принудительного выбора
         единственного токена не подходит как confidence.

    Параметры
    ----------
    text : str
        Пользовательский запрос, который нужно классифицировать.
    max_new_tokens : int, optional
        Сколько токенов максимум разрешаем модели сгенерировать
        (включая JSON и возможный «хвост»).  Дефолт — 32.

    Возвращает
    ----------
    label : str
        Предсказанная категория из `label_names`; None при ошибке формата.
    confidence : float
        Уверенность модели (0 – 1).  Расчёт: exp(среднего log p токенов метки).
    corrupted : int
        0 — формат корректный и label ∈ `label_names`;
        1 — формат сломан **или** label не из списка.
    raw_generation : str
        Полный ответ LLM (полезно для дебага/визуализации).
    """
    return
# ---- Конец кода ----



In [ ]:
# ---- Ваш код здесь ----
print("""
1. Посчитайте roc_auc_score(y_true_bin, y_score), где
y_true_bin = 1, если модель угадала (`pred_label == true_label`)), иначе 0, y_score = confidence, который вернула annotate_conf
2. Сделайте выводы по полезности confidence
""")
# ---- Конец кода ----


## 8.1. Human-in-the-loop с шумным разметчиком

В реальных задачах разметку часто делают не идеальные эксперты, а обычные асессоры — они тоже ошибаются. Для достижения хорошего качества с ними используется разметка с перкрытием по N accесорам, а затем агрегируется разметка (например majority vote). Таким образом получаем **агрегированную метку** и меру согласованности ассесоров - насколько они сходятся в решении (например 0.75 - из 4 ассессоров 3 выбрали итоговую метку).

Согласованность не гарантирует правильность: чтобы использовать разметку как gold, нужна дополнительная проверка качества.

Часть разметки можно переложить на LLM, если уметь оценивать его уверенность.

В этом задании мы смоделируем простую схему:

- есть **gold-метка** `true_label` (используем только для оценки качества),
- есть один **"человек"-разметчик** с ошибками (`human_label`),
- есть предсказания LLM: `pred_label` и `confidence`.

Мы хотим построить **гибридную систему**:

- по умолчанию используем метку человека (`human_label`);
- если `confidence >= threshold` — считаем, что LLM очень уверен и берём его метку (`pred_label`).

#### 8.2.1. Симуляция "шумного" разметчика (3 балла)

Реализуйте функцию, которая по gold-меткам `true_labels` возвращает `human_labels`:

- с вероятностью `1 - error_rate` берётся правильная метка,
- с вероятностью `error_rate` — случайная *другая* метка из `label_names`  
  (можно взять, например, `error_rate = 0.20`).

1. Реализуйте функцию `simulate_noisy_human(true_labels, label_names, error_rate=0.20, random_state=42)`.
2. Посчитайте accuracy такого разметчика относительно `true_labels`.

Замечание: в реальных задачах расхождения между разметчиками и ошибочно проставленные метки зачастую не случайны. В данном задании симулируем шум для упрощения.
Сгенерируйте human_labels один раз и используйте их при всех порогах. Это вероятность ошибки 20%, а не требование получить ровно 20% ошибочных строк.

Здесь 20% — выбранный сценарий ошибки одного разметчика, а не измеренное качество людей на Banking77 и не процент несогласия между ними. В исходном примере COLING 2025 шум зависел от согласованности разметки каждого объекта; здесь используем одну вероятность для простоты.


In [ ]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.2. Гибридная схема разметки (4 балла)

Реализуйте функцию, которая комбинирует разметку человека и LLM по порогу уверенности:

- по умолчанию использует `human_labels`,
- если `pred_labels[i]` — допустимая метка и `confidence[i] >= threshold` — вместо метки человека берёт `pred_labels[i]`; невалидный ответ всегда остаётся человеку,
- возвращает:
  - `overall_acc` — итоговую accuracy гибридной разметки (относительно `true_labels`),
  - `coverage` — долю объектов, где использовали LLM (от 0 до 1).

1. Реализуйте функцию  
   `simulate_hybrid(pred_labels, human_labels, true_labels, confidence, threshold)`.
2. Проверьте её на небольшом игрушечном примере

In [ ]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.3. Подбор порога и анализ trade-off (3 балла)

1. Для `threshold` из диапазона `[0.0, 1.0]` с шагом 0.01:
   - посчитайте `overall_acc` и `coverage`;
   - выведите таблицу с колонками `threshold`, `accuracy`, `coverage`  
     (по желанию можно дополнительно построить график `accuracy` vs `coverage`).
2. Найдите значение `threshold`, при котором:
   - качество гибридной разметки **не ниже** заданного уровня (например, accuracy ≥ 0.8 *относительно gold*),
   - а `coverage` максимально возможен.
Если допустимого порога нет, явно сообщите об этом; это тоже корректный результат. Подбор на этой же выборке — учебное описательное сравнение, а не независимая оценка выбранного порога.


In [ ]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


#### 8.2.4. Сравнение схем разметки (2 балла)

1. Посчитайте и сравните:
   - accuracy только человека (`human_labels`),
   - accuracy только LLM (`pred_labels`),
   - лучшую точку гибрида.
2. Кратко (2–3 предложения) ответьте:
   - появилось ли преимущество гибрида: как изменились accuracy и доля запросов человеку относительно «чисто человек» и «чисто LLM»?
   - в каких кейсах такая схема human-in-the-loop может быть особенно полезна?

Перед сдачей сохраните ноутбук с outputs. В конце покажите таблицу экспериментов, accuracy/AUC confidence и таблицу выбора порога; укажите модель, размер выборки и seed.


In [ ]:
# ---- Ваш код здесь ----

# ---- Конец кода ----


## Итоги домашки

В этой работе мы посмотрели на разметку как на систему, где есть и люди, и LLM.

Главное, что нужно вынести:
- LLM можно использовать как разметчика (при этом важно следить за качеством ), можно улучшать промт за счет различных прдеставленных способов.
- Оценку **уверенности** по logprobs нужно проверить на данных, прежде чем решать, где доверять модели, а где подключать человека.
- Гибридную схему human-in-the-loop сравниваем с «только крауд» и «только LLM» по качеству и доле автоматизации: выигрыш не гарантирован.
- Эти идеи масштабируются дальше: улучшение промптов, дообучение модели, active learning и более умные пайплайны разметки.
